# Task 2 — Model & Feature-Set Comparison

Goal: find the best **(model × feature set)** combination, then fully evaluate the winner.

- **Models:** Logistic Regression, Random Forest, XGBoost
- **Feature sets:** `lean (9)`, `balanced (11)`, `full (19)` — the three we shortlisted
- **Method:** split once, compare all 9 combinations by **5-fold cross-validation on the training data**, pick the winner, then evaluate it **once** on the held-out test set (the honest number).

This notebook is self-contained — it rebuilds the features, so run it top to bottom.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def _find_data_dir():
    for d in ("datasets", "../datasets"):
        if os.path.isfile(os.path.join(d, "studentInfo.csv")):
            return d
    raise FileNotFoundError("Missing datasets/. Place OULAD CSV files in <project>/datasets/.")

DATA_DIR = _find_data_dir()
print("using data folder:", DATA_DIR)

WEEK6 = 42
KEY = ["code_module", "code_presentation", "id_student"]

student_info = pd.read_csv(f"{DATA_DIR}/studentInfo.csv")
student_vle  = pd.read_csv(f"{DATA_DIR}/studentVle.csv")
student_reg  = pd.read_csv(f"{DATA_DIR}/studentRegistration.csv")
assessments  = pd.read_csv(f"{DATA_DIR}/assessments.csv")
student_ass  = pd.read_csv(f"{DATA_DIR}/studentAssessment.csv")
vle          = pd.read_csv(f"{DATA_DIR}/vle.csv")
print("loaded")

## Rebuild the 19 features + target
(Same feature engineering as the previous notebook — condensed here so this one runs on its own.)

In [ ]:
labels = student_info[KEY + ["final_result"]].copy()
labels["at_risk"] = labels["final_result"].isin(["Withdrawn", "Fail"]).astype(int)

vle_win = student_vle[(student_vle["date"] >= 0) & (student_vle["date"] < WEEK6)].copy()
vle_pre = student_vle[student_vle["date"] < 0].copy()

base = (vle_win.groupby(KEY)
        .agg(total_clicks=("sum_click", "sum"),
             active_days=("date", "nunique"),
             distinct_materials=("id_site", "nunique")).reset_index())
vle_win["wk"] = vle_win["date"] // 7
weeks = vle_win.groupby(KEY)["wk"].nunique().rename("active_weeks").reset_index()
daily = vle_win.groupby(KEY + ["date"])["sum_click"].sum().reset_index()
reg_std = daily.groupby(KEY)["sum_click"].std().rename("clicks_std").reset_index()
last_day = daily.groupby(KEY)["date"].max().rename("last_active_day").reset_index()
early = (vle_win[vle_win["date"] < 14].groupby(KEY)["sum_click"].sum()
         .rename("clicks_weeks_1_2").reset_index())
late  = (vle_win[vle_win["date"] >= 28].groupby(KEY)["sum_click"].sum()
         .rename("clicks_weeks_5_6").reset_index())
pre = vle_pre.groupby(KEY)["sum_click"].sum().rename("pre_start_clicks").reset_index()

vle_typed = vle_win.merge(
    vle[["id_site", "code_module", "code_presentation", "activity_type"]],
    on=["id_site", "code_module", "code_presentation"], how="left")
def type_sum(types, name):
    return (vle_typed[vle_typed["activity_type"].isin(types)]
            .groupby(KEY)["sum_click"].sum().rename(name).reset_index())
forum   = type_sum(["forumng", "ouwiki", "oucollaborate"], "forum_clicks")
quiz    = type_sum(["quiz", "externalquiz", "questionnaire"], "quiz_clicks")
content = type_sum(["oucontent", "resource", "subpage", "url", "page", "homepage"], "content_clicks")

early_assess = (assessments[assessments["date"] < WEEK6][["id_assessment", "date"]]
                .rename(columns={"date": "due_date"}))
ass = student_ass.merge(early_assess, on="id_assessment", how="inner")
ass["days_early"] = ass["due_date"] - ass["date_submitted"]
ass_feat = (ass.groupby("id_student")
            .agg(num_assess_submitted=("id_assessment", "nunique"),
                 mean_assess_score=("score", "mean"),
                 mean_days_early=("days_early", "mean")).reset_index())

reg  = student_reg[KEY + ["date_registration"]].copy()
info = student_info[KEY + ["num_of_prev_attempts", "studied_credits"]].copy()

features = labels[KEY].copy()
for part in [base, weeks, reg_std, last_day, early, late, pre, forum, quiz, content, reg, info]:
    features = features.merge(part, on=KEY, how="left")
features = features.merge(ass_feat, on="id_student", how="left")
features["days_since_last_active"] = (41 - features["last_active_day"]).fillna(42)
features = features.drop(columns=["last_active_day"]).fillna(0)
features["engagement_trend"] = features["clicks_weeks_5_6"] - features["clicks_weeks_1_2"]
features = features.merge(labels[KEY + ["at_risk"]], on=KEY, how="left")

FEATURE_COLS = [
    "total_clicks", "active_days", "active_weeks", "distinct_materials",
    "clicks_std", "days_since_last_active",
    "clicks_weeks_1_2", "clicks_weeks_5_6", "engagement_trend", "pre_start_clicks",
    "forum_clicks", "quiz_clicks", "content_clicks",
    "num_assess_submitted", "mean_assess_score", "mean_days_early",
    "date_registration", "num_of_prev_attempts", "studied_credits",
]
print(f"rebuilt {len(FEATURE_COLS)} features for {len(features):,} students")

## Define the three feature sets + three models

In [ ]:
from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier   # !pip install xgboost  (if needed)

# --- the three shortlisted feature sets ---
core = ["days_since_last_active", "active_days", "distinct_materials",
        "mean_assess_score", "clicks_std", "quiz_clicks", "clicks_weeks_5_6"]
lean     = core + ["studied_credits", "num_assess_submitted"]                       # 9
balanced = core + ["num_assess_submitted", "pre_start_clicks",
                   "studied_credits", "num_of_prev_attempts"]                       # 11
full     = FEATURE_COLS                                                             # 19
feature_sets = {"lean (9)": lean, "balanced (11)": balanced, "full (19)": full}

# --- split ONCE; all comparison uses CV on train, test stays sacred ---
y = features["at_risk"]
train_df, test_df = train_test_split(features, test_size=0.25,
                                     random_state=42, stratify=y)
y_train = train_df["at_risk"]
y_test  = test_df["at_risk"]
neg, pos = (y_train == 0).sum(), (y_train == 1).sum()

# fresh models each time so nothing carries state between runs
def make_models():
    return {
        "LogReg": make_pipeline(
            StandardScaler(),
            LogisticRegression(max_iter=1000, class_weight="balanced")),
        "RandomForest": RandomForestClassifier(
            n_estimators=300, class_weight="balanced", random_state=42, n_jobs=-1),
        "XGBoost": XGBClassifier(
            n_estimators=300, max_depth=5, learning_rate=0.1,
            scale_pos_weight=neg / pos, eval_metric="logloss", random_state=42),
    }
print("ready")

## The 3 x 3 comparison (cross-validated on training data)
This trains every model on every feature set with 5-fold CV. It may take a minute. Higher is better; we read **ROC-AUC** as the main score and **Recall** alongside.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rows = []
for fs_name, cols in feature_sets.items():
    for m_name, model in make_models().items():
        res = cross_validate(model, train_df[cols], y_train, cv=cv,
                             scoring=["roc_auc", "recall"])
        rows.append({"feature_set": fs_name, "model": m_name,
                     "AUC": res["test_roc_auc"].mean(),
                     "Recall": res["test_recall"].mean()})
grid = pd.DataFrame(rows)

auc_table = grid.pivot(index="model", columns="feature_set", values="AUC").round(4)
rec_table = grid.pivot(index="model", columns="feature_set", values="Recall").round(4)
print("ROC-AUC  (5-fold CV on train) — main score:")
print(auc_table.to_string())
print("\nRecall  (5-fold CV on train):")
print(rec_table.to_string())

**How to choose:** find the highest AUC cell. If a *leaner* set is within ~0.005 of the best, prefer it (simpler + more interpretable, and you can defend that in the report). Note your pick, then fill it in below.

## Final evaluation of the winner (run ONCE)
Edit the two lines below to your chosen combo, then run. This is the only time the **test set** is used — these are the numbers for your report.

In [ ]:
from sklearn.metrics import (precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix)

# <-- EDIT these two after reading the grid -->
BEST_FEATURES   = full          # lean / balanced / full
BEST_MODEL_NAME = "XGBoost"     # "LogReg" / "RandomForest" / "XGBoost"

final_model = make_models()[BEST_MODEL_NAME]
final_model.fit(train_df[BEST_FEATURES], y_train)
proba = final_model.predict_proba(test_df[BEST_FEATURES])[:, 1]

print(f"FINAL: {BEST_MODEL_NAME} on {len(BEST_FEATURES)} features — held-out test set\n")
for t in [0.50, 0.35]:
    pred = (proba >= t).astype(int)
    print(f"threshold {t}:  recall {recall_score(y_test, pred):.3f}   "
          f"precision {precision_score(y_test, pred):.3f}   "
          f"F1 {f1_score(y_test, pred):.3f}   AUC {roc_auc_score(y_test, proba):.3f}")
    print("  confusion matrix:", confusion_matrix(y_test, pred).tolist())

### Calibration of the final model
Does a predicted 70% risk correspond to ~70% actual withdrawals? Closer to the diagonal = better.

In [ ]:
from sklearn.calibration import calibration_curve

frac_pos, mean_pred = calibration_curve(y_test, proba, n_bins=10)
plt.figure(figsize=(6, 6))
plt.plot([0, 1], [0, 1], "k--", label="Perfectly calibrated")
plt.plot(mean_pred, frac_pos, "o-", label=BEST_MODEL_NAME)
plt.xlabel("Predicted risk"); plt.ylabel("Actual withdrawal rate")
plt.title("Calibration of final model"); plt.legend(); plt.tight_layout()
plt.savefig("final_calibration.png", dpi=150); plt.show()

### Top features of the final model + threshold sweep

In [ ]:
# Feature importance (works for tree models; falls back to coefficients for LogReg)
est = final_model.steps[-1][1] if hasattr(final_model, "steps") else final_model
if hasattr(est, "feature_importances_"):
    imp = pd.Series(est.feature_importances_, index=BEST_FEATURES)
else:
    imp = pd.Series(np.abs(est.coef_[0]), index=BEST_FEATURES)

print("Top 3 features driving risk:")
for i, (f, s) in enumerate(imp.sort_values(ascending=False).head(3).items(), 1):
    print(f"  {i}. {f}  ({s:.3f})")

plt.figure(figsize=(7, 5))
imp.sort_values().plot(kind="barh")
plt.title("What drives the final model's risk prediction")
plt.xlabel("importance"); plt.tight_layout()
plt.savefig("final_feature_importance.png", dpi=150); plt.show()

In [ ]:
# Threshold sweep — justify the operating threshold
thresholds = np.arange(0.05, 0.96, 0.05)
prec, rec, f1s = [], [], []
for t in thresholds:
    pred = (proba >= t).astype(int)
    prec.append(precision_score(y_test, pred, zero_division=0))
    rec.append(recall_score(y_test, pred))
    f1s.append(f1_score(y_test, pred))

plt.figure(figsize=(8, 5))
plt.plot(thresholds, prec, "o-", label="Precision")
plt.plot(thresholds, rec,  "s-", label="Recall")
plt.plot(thresholds, f1s,  "^-", label="F1")
plt.axvline(0.35, color="gray", ls="--", label="candidate (0.35)")
plt.xlabel("Threshold"); plt.ylabel("Score")
plt.title(f"{BEST_MODEL_NAME}: metrics vs threshold")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig("final_threshold_sweep.png", dpi=150); plt.show()

---
After running, note the highest AUC cell in the grid, set `BEST_FEATURES` and `BEST_MODEL_NAME` below, then re-run the final evaluation cells for report numbers.